# Tutorial 60 — Create embeddings

[Tutorial 50](./50_source_structure.ipynb) produced searchable chunks. This
tutorial uses an **embedding model** to convert those chunks, and then a
question, into equal-length numeric vectors.

You will see the difference between words and **model tokens**, document
embeddings and query embeddings, and raw vectors versus similarity scores. The tutorial
also makes the Qwen3 query-formatting contract explicit: query formatting belongs
to the embedding model, not to semantic retrieval itself.

Nothing is indexed or retrieved yet.
[Tutorial 70](./70_semantic_retrieval.ipynb) turns these vectors into a
reusable semantic retrieval index.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import qachat


## 2. Load an embedding model

`auto` resolves the computational backend first; the corresponding canonical
QAChat embedding-model default is then selected. This prevents an MLX model ID
from being paired accidentally with PyTorch, or vice versa. The later chat model
used for answer generation is a separate model with a separate purpose.


In [ ]:
REQUESTED_BACKEND = qachat.BACKEND_AUTO

SELECTED_BACKEND = qachat.choose_backend(REQUESTED_BACKEND)

MODEL_ID = (
    qachat.DEFAULT_EMBEDDING_MODEL_MLX
    if SELECTED_BACKEND == "mlx"
    else qachat.DEFAULT_EMBEDDING_MODEL_TORCH
)

print(f"Requested backend: {REQUESTED_BACKEND}")
print(f"Resolved backend: {SELECTED_BACKEND}")
print(f"Embedding model: {MODEL_ID}")


In [ ]:
ALLOW_DOWNLOAD = True


In [ ]:
embedding_session = qachat.EmbeddingSession(
    MODEL_ID,
    backend=SELECTED_BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 3. Prepare repository chunks

The following cells repeat the file-selection, source-unit, and chunking pipeline
from [Tutorials 40](./40_repository_files.ipynb) and
[50](./50_source_structure.ipynb) so this notebook remains independently
runnable. This time chunk sizes are measured with the actual embedding tokenizer.


In [ ]:
report = qachat.scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat/repository.py"],
)

print(report["selected_files"][0])


In [ ]:
source_files = qachat.read_repository_files(report)
units = qachat.extract_source_units(source_files)

print(f"Natural units: {len(units)}")


In [ ]:
chunks = qachat.chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=120,
    max_tokens=160,
    overlap_tokens=20,
)

print(f"Chunks: {len(chunks)}")


## 4. Compare words with model tokens

A model token can be a whole word, part of a word, punctuation, or a code
symbol. The embedding model's context limit is defined in tokens, not words, so
QAChat uses the model tokenizer when deciding how much source text fits in a
chunk.


In [ ]:
first_chunk = chunks[0]

print(f"{first_chunk}\n\n{first_chunk.text}")


In [ ]:
word_count = len(first_chunk.text.split())

print(f"Visible words: {word_count}")


In [ ]:
token_count = embedding_session.count_tokens(first_chunk.text)

print(f"Model tokens: {token_count}")


## 5. Embed repository chunks

Each chunk becomes one embedding vector. Vectors produced by the same model have
the same dimensionality. Their numeric components are learned features; they are
not intended to be interpreted as readable words or hand-designed properties.


In [ ]:
selected_chunks = chunks
texts = [chunk.text for chunk in selected_chunks]

print(f"Chunks sent: {len(selected_chunks)}")

for i, text in enumerate(texts):
    print(f"\n>> CHUNK {i} <<\n\n{text}\n")


In [ ]:
document_vectors = embedding_session.embed_documents(texts)

print(f"Vectors returned: {len(document_vectors)}")

for i, document_vector in enumerate(document_vectors):
    print(f"\n>> VECTOR {i} <<\n\n{document_vector}\n")


In [ ]:
dimensions = {len(vector) for vector in document_vectors}

assert len(document_vectors) == len(selected_chunks)
assert len(dimensions) == 1

dimension = dimensions.pop()

assert dimension > 0

print(f"Vector length: {dimension}")


In [ ]:
for chunk, vector in zip(
    selected_chunks,
    document_vectors,
    strict=True,
):
    name = chunk.symbol or "(module context)"
    print(name, vector)


## 6. Embed a question

The canonical QAChat embedding default is Qwen3 Embedding. Qwen3 expects a query
to include an explicit retrieval instruction, so the tutorial formats the user
question before embedding it.

This formatting is part of the **embedding-model contract**, not part of
semantic retrieval itself. A model that expects plain queries would receive the
question directly while the retrieval algorithm could remain unchanged.


In [ ]:
QUESTION = "Which function scans a repository and selects files by extension?"

query_text = qachat.format_qwen3_query(
    instruction=qachat.SOFTWARE_REPOSITORY_INSTRUCTION,
    question=QUESTION,
)

print(query_text)


In [ ]:
question_vector = embedding_session.embed_query(query_text)

assert len(question_vector) == dimension

print(f"Question vector length: {len(question_vector)}\n")
print(question_vector)


This format belongs to Qwen3 Embedding, not every embedding model. For a model that expects a plain question, use `query_text = QUESTION`. The reusable `EmbeddingSession` remains model-neutral.

## 7. Compare question and chunk embeddings

**Cosine similarity** measures the relative direction of two vectors. A
larger value means the question and chunk embeddings are more similar under this model. Here
the comparison is performed directly so the mechanism is visible;
[Tutorial 70](./70_semantic_retrieval.ipynb) wraps the stored document vectors
in a reusable semantic index.


In [ ]:
def cosine_similarity(first: list[float], second: list[float]) -> float:
    dot_product = sum(
        first_value * second_value
        for first_value, second_value in zip(first, second, strict=True)
    )
    first_length = sum(value**2 for value in first) ** 0.5
    second_length = sum(value**2 for value in second) ** 0.5
    return dot_product / (first_length * second_length)


In [ ]:
ranked_chunks = sorted(
    (
        (cosine_similarity(question_vector, vector), chunk)
        for chunk, vector in zip(
            selected_chunks,
            document_vectors,
            strict=True,
        )
    ),
    key=lambda result: result[0],
    reverse=True,
)


In [ ]:
for score, chunk in ranked_chunks:
    name = chunk.symbol or "(module context)"
    print(f"{score:.3f} | {name}")


In [ ]:
best_score, best_chunk = ranked_chunks[0]
best_name = best_chunk.symbol or "(module context)"

print(f"Most relevant chunk: {best_score:.3f} | {best_name}\n")
print(best_chunk.text)


## 8. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Continue the tutorial series

**Previous:** [Tutorial 50 — Structure source files into chunks](./50_source_structure.ipynb)

**Next:** [Tutorial 70 — Retrieve by semantic similarity](./70_semantic_retrieval.ipynb)
